In [ ]:
from cdo import Cdo
import xarray as xr
import pandas as pd
import glob
import os 

os.chdir('../..') # beginning of the repository

%load_ext autoreload
%autoreload 2

In [2]:
esm_log = pd.read_csv('data/logs/esm_log_pr.csv')

# Precipitation remapping

In [4]:
source_path = 'data/reanalysis/prect.mon.mean.72x144.nc'

cdo = Cdo()
PRECT_obs = xr.open_dataset(source_path)
nlat = PRECT_obs.latitude.size
nlon = PRECT_obs.longitude.size

try:
    grid_path = f'data/grids/pr_obs_{nlat}x{nlon}.txt'
    grid_info = cdo.griddes(input = source_path)
    # Save the string to a file or process it
    with open(grid_path, 'w') as f:
        for line in grid_info:
            f.write(line + '\n')
except Exception as e:
    print(f"Error: {e}")

nlat, nlon

(72, 144)

In [9]:
esm_subset = esm_log[(esm_log['Variable ID'] == 'pr') & (esm_log['Lat'] >= nlat) & (esm_log['Lon'] >= nlon)]
esm_subset.head()

,ESM,Experiment ID,Variable ID,Member ID,Lat,Lon,Grid Points
0,ACCESS-CM2,historical,pr,r1i1p1f1,144,192,27648
1,ACCESS-ESM1-5,historical,pr,r1i1p1f1,145,192,27840
2,AWI-CM-1-1-MR,historical,pr,r1i1p1f1,192,384,73728
3,AWI-ESM-1-1-LR,historical,pr,r1i1p1f1,96,192,18432
4,BCC-CSM2-MR,historical,pr,r1i1p1f1,160,320,51200


In [ ]:
var = 'pr'
target_grid = 'data/grids/pr_obs_72x144.txt'
output_path_base = f'data/CMIP6_monthly_data/remapped/{var}'
os.makedirs(output_path_base, exist_ok=True)

for idx, esm_info in esm_subset.iterrows():
    print(f'Processing : {esm_info['ESM']}')

    input_path = glob.glob(f'data/CMIP6_monthly_data/r1i1p1f1/{var}/historical/{esm_info['ESM']}*_historical_r1i1p1f1_{var}.nc')
    output_path = os.path.join(output_path_base, f'{esm_info['ESM']}_{var}_{nlat}x{nlon}_remapped.nc')

    if os.path.exists(os.path.join(output_path)):
        print(f'    File {os.path.basename(output_path)} already existing ...')
        pass
    elif len(input_path) == 0:
        print(f'    File {os.path.basename(output_path)} not found ...')
        continue
    else:
        try:
            remapped_data = cdo.remapcon(target_grid, input=input_path[0], output=output_path) #, options = '-f nc')
        except:
            ValueError(f'Unable to remap {os.path.basename(input_path)} ...')
            continue
    #break

Processing : ACCESS-CM2
    File ACCESS-CM2_pr_72x144_remapped.nc already existing ...
Processing : ACCESS-ESM1-5
    File ACCESS-ESM1-5_pr_72x144_remapped.nc already existing ...
Processing : AWI-CM-1-1-MR
    File AWI-CM-1-1-MR_pr_72x144_remapped.nc already existing ...
Processing : AWI-ESM-1-1-LR
    File AWI-ESM-1-1-LR_pr_72x144_remapped.nc already existing ...
Processing : BCC-CSM2-MR
    File BCC-CSM2-MR_pr_72x144_remapped.nc already existing ...
Processing : CAMS-CSM1-0
    File CAMS-CSM1-0_pr_72x144_remapped.nc already existing ...
Processing : CAS-ESM2-0
    File CAS-ESM2-0_pr_72x144_remapped.nc not found ...
Processing : CESM2
    File CESM2_pr_72x144_remapped.nc already existing ...
Processing : CESM2-FV2
    File CESM2-FV2_pr_72x144_remapped.nc already existing ...
Processing : CESM2-WACCM
    File CESM2-WACCM_pr_72x144_remapped.nc already existing ...
Processing : CESM2-WACCM-FV2
    File CESM2-WACCM-FV2_pr_72x144_remapped.nc already existing ...
Processing : CMCC-CM2-HR4
